# torchvisionによるモデル呼び出しと事前学習モデルの利用

---
## 目的
`torchvision.models`を用いて，ResNet50・EfficientNetなど代表的な画像分類モデルを呼び出す方法と，ImageNetで学習済みの重み（事前学習モデル）を読み込んで推論を行う方法について理解する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
import os
import urllib.request

import torch
import torch.nn.functional as F
import torchvision
from torchvision.models import resnet50, ResNet50_Weights, efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## torchvision.modelsによるモデルの呼び出し
`torchvision.models`には，これまでのノートブックでスクラッチ実装してきたResNetやEfficientNetをはじめ，多数の代表的な画像分類モデルがあらかじめ実装されています．`torchvision.models.list_models()`を用いることで，現在利用可能なモデルの一覧を確認できます．

In [ ]:
all_models = torchvision.models.list_models()
print('利用可能なモデル数:', len(all_models))

# 名前に'resnet'を含むモデルの一覧
resnet_models = [m for m in all_models if 'resnet' in m]
print('ResNet系のモデル:', resnet_models)

## 事前学習済み重み（Weights API）
torchvisionのモデルは，`<モデル名>_Weights`という列挙型（Enum）を通じて，ImageNetなどのデータセットで学習済みの重みを指定できます．例えば`ResNet50_Weights`には，学習方法の異なる複数の重み（`IMAGENET1K_V1`, `IMAGENET1K_V2`など）が用意されています．

各重みは，学習時に使用したものと同じ前処理（リサイズ・クロップ・正規化）を`transforms()`メソッドで取得でき，またクラス名の一覧を`meta['categories']`から参照できます．学習時と異なる前処理を行うと精度が大きく低下するため，`weights.transforms()`を使うことが推奨されます．

In [ ]:
weights = ResNet50_Weights.IMAGENET1K_V2

print('クラス数:', len(weights.meta['categories']))
print('クラス名の例:', weights.meta['categories'][:5])
print(weights.transforms())

## サンプル画像の準備
推論に使用するサンプル画像をダウンロードします．

In [ ]:
img_path = 'dog.jpg'
if not os.path.exists(img_path):
    urllib.request.urlretrieve('https://github.com/pytorch/hub/raw/master/images/dog.jpg', img_path)

image = Image.open(img_path).convert('RGB')

plt.imshow(image)
plt.axis('off')
plt.show()

## ResNet50による推論
`weights`を指定してモデルを作成すると，指定した事前学習済み重みが自動的にダウンロード・読み込みされます．推論時には，学習済みのBatch Normalization・Dropoutの統計量をそのまま使用するため，必ず`model.eval()`でモデルを評価モードに切り替えます．

In [ ]:
weights = ResNet50_Weights.IMAGENET1K_V2
model = resnet50(weights=weights)
model = model.to(device)
model.eval()  # 事前学習モデルを推論に使う際は必ずevalモードにする

preprocess = weights.transforms()  # 学習時と同じ前処理（リサイズ・正規化など）
input_tensor = preprocess(image).unsqueeze(0).to(device)  # バッチ次元を追加

with torch.no_grad():
    output = model(input_tensor)
probs = F.softmax(output[0], dim=0)

categories = weights.meta['categories']
top5_prob, top5_idx = torch.topk(probs, 5)
print('ResNet50の予測結果（上位5クラス）')
for prob, idx in zip(top5_prob, top5_idx):
    print(f'  {categories[idx]}: {prob.item():.4f}')

## EfficientNetによる推論
モデルの種類が変わっても，「`<モデル名>_Weights`を指定してモデルを作成する」という手順は共通です．ここでは，`efficientnet_b0`を用いて同じ画像を分類してみます．

In [ ]:
weights_effnet = EfficientNet_B0_Weights.IMAGENET1K_V1
model_effnet = efficientnet_b0(weights=weights_effnet)
model_effnet = model_effnet.to(device)
model_effnet.eval()

preprocess_effnet = weights_effnet.transforms()
input_tensor_effnet = preprocess_effnet(image).unsqueeze(0).to(device)

with torch.no_grad():
    output_effnet = model_effnet(input_tensor_effnet)
probs_effnet = F.softmax(output_effnet[0], dim=0)

categories_effnet = weights_effnet.meta['categories']
top5_prob_effnet, top5_idx_effnet = torch.topk(probs_effnet, 5)
print('EfficientNet-B0の予測結果（上位5クラス）')
for prob, idx in zip(top5_prob_effnet, top5_idx_effnet):
    print(f'  {categories_effnet[idx]}: {prob.item():.4f}')

## モデル出力層の確認
事前学習済みモデルをファインチューニングする際には，最終層（分類ヘッド）を差し替える必要があります．そのため，最終層の名前と構造をあらかじめ確認しておきます．ResNet系は`model.fc`，EfficientNet系は`model.classifier`という名前の層が最終的な全結合層に対応しています．

In [ ]:
print('ResNet50の最終層 (model.fc):')
print(model.fc)
print()
print('EfficientNet-B0の最終層 (model.classifier):')
print(model_effnet.classifier)

## 課題

1. 他のモデル（例：`vgg16`, `densenet121`, `vit_b_16`）を`torchvision.models`から読み込み，同じ画像に対する予測結果を比較しましょう．

2. 自分で用意した別の画像を使って推論を行い，予測結果がどのように変化するか確認しましょう．

3. `ResNet50_Weights`には`IMAGENET1K_V1`と`IMAGENET1K_V2`の2種類があります．両方の重みで推論を行い，予測結果や信頼度がどのように異なるか比較しましょう．